[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fw-ai/cookbook/blob/main/training/case-studies/dpo_antislop/dpo_glm53_flash.ipynb)

On Colab, run the clone-and-chdir block at the top of the Setup cell (section 1) before anything else: the recorded cells read files from the checked-out repo.

# DPO from scratch on GLM 5.3 Flash, serverless

A LoRA adapter on GLM 5.3 Flash that learns from pairs of social posts ranked by an AI-text detector, trained by a Direct Preference Optimization (DPO) loop you build by hand on the serverless Training API.

The headline comparison: on 20 held-out briefs the DPO adapter scored 0.30 on the detector [0.17, 0.44] against the base model's 0.33 [0.21, 0.46], so DPO alone did not move it; 20 steps of RL on the same detector reached 0.79 [0.69, 0.87] and left 4 distinct openers in 40 posts. The figure is in [See the result first](#See-the-result-first), just below.

**Needs:** Python 3.12 (3.11 works; 3.10 fails on import). From `cookbook/training`: `uv venv --python 3.12 && source .venv/bin/activate && uv pip install -e . && uv pip install ipykernel matplotlib openai`. Open Jupyter from this folder (`case-studies/dpo_antislop`). The notebook imports `../common/figures.py` and this folder's `antislop_figures.py` and `antislop_data.py`.

**Status:** Recorded results are read from committed files; Run All makes no API call and spends nothing. The first run downloads UltraFeedback (about 1 MB of pairs) and the GLM 5.3 Flash tokenizer from Hugging Face: network, not money, and both are cached afterwards. Without a network the data cell falls back to the committed Anti-slop pack and the rendering checks are skipped, with a message in each case. The paid knobs, all off by default:

| knob | where | what it costs | needs |
| --- | --- | --- | --- |
| `RUN_LIVE = True` | config cell, section 1 | runs every `%%live` cell. The default smoke run is 10 steps over 200 pairs plus a 32-prompt win-rate: minutes of trainer time, cost not measured. The full run (`STEPS = 370`, `MAX_PAIRS = 3000`) is 2.5-3 hours, about 2.4M training tokens and 2.8M prefill tokens at the [serverless training rates](https://fireworks.ai/pricing); the detailed guide (`GUIDE.md`)'s cost section has the breakdown | `FIREWORKS_API_KEY` (account-scoped) |
| `GENERATE_WITH_REASONING = True` | config cell | sections 10-11 generate with the think block open: 30-60 s per generation, a win-rate of an hour or more, billed as sampling tokens | `RUN_LIVE` |
| `EVAL_PAIRS` (and `N_WINRATE`) | config cell | the win-rate size: 2 generations and 2 judge calls per prompt, judge calls billed as ordinary serverless inference | `RUN_LIVE` |
| `RESUME_FROM` | config cell | continues an earlier run from one of its checkpoints; the same per-step cost from there | `RUN_LIVE` |
| `PROMOTE = True` | section 12 | registers the adapter as a model in your account under `OUTPUT_MODEL_ID`. Registering is free; nothing runs until you deploy it | `RUN_LIVE`, `OUTPUT_MODEL_ID` |
| `DATA_JSONL` | config cell | free: points the loop at your own pairs or the committed pack | nothing |

`FIREWORKS_BASE_URL` is optional: it only changes which Fireworks endpoint the key talks to (default `https://api.fireworks.ai`). Leave it unset.

**Three routes:** Explore the result ("See the result first", no key) · Reproduce it (sections 1-4 and 6-13 with `RUN_LIVE = True`, paid) · Use your own data (set `DATA_JSONL` in section 1; section 2 validates it).

**Route map.** Quick path: "See the result first", then sections 1, 2, 3, 4, 6, 7, 8, 9, 10, 11, 12 and 13. Method deep dive, each marked "(deep dive, skippable)": section 5 (the loss, and the trap the reference does not prevent), "The training boundary" and "Resumable checkpoints" inside section 6. Cells marked **Recorded** read committed files; cells marked **Live** start with `%%live` and produce output only when `RUN_LIVE = True`.

**What this notebook is.** The method behind the gallery's Anti-slop adapter: a complete DPO loop, the reference model, the loss and the training step, built by hand on `glm-5p3-flash`, with nothing to provision and nothing to tear down. The method is fixed; the data is pluggable. The demo pairs are UltraFeedback, a public general-helpfulness preference set, so Run All works for anyone. The gallery's own adapter trains the same loop on a frozen pack of chosen/rejected social-post pairs; point `DATA_JSONL` at that file and nothing else changes.

**Not this notebook if you just want to ship.** Managed DPO jobs ([`../dpo_style/`](../dpo_style/), `client.dpo_jobs`) do all of this server-side in four SDK calls. Read that one to *use* DPO; read this one to *understand* it.

**What DPO is.** You show the model pairs of answers to the same prompt plus a verdict on which one a person preferred. No reward function, no reward model, and no sampling during training.

| | needs | generates during training? |
| --- | --- | --- |
| **SFT** | one ideal answer per prompt | no |
| **DPO** | two answers + which is better | **no** |
| **RL (GRPO)** | a prompt + a reward *function* | yes, constantly |

Cheaper and more stable than RL, but its ceiling is your labels: it never explores.

**One thing to know about GLM 5.3 Flash.** It is a reasoning model: by default it thinks inside a `<think>` block before answering. Preference pairs rarely carry reasoning, and the renderer handles that by writing an *empty* think block in front of every training response, on the prompt side, where it carries no gradient. Section 6 shows exactly where the loss starts, and section 10 explains what that means when you generate from the tuned model.

**Why serverless.** Nothing to provision, no GPU quota. That matters here because DPO needs a *second* frozen model, and serverless gives you one as a snapshot instead of a second GPU job.


### Glossary

- **Adapter / LoRA rank.** A LoRA adapter is a small set of extra weights trained on top of a frozen base model; the rank (64 here) is the width of those weights and sets the adapter's size.
- **Zero-padded rank 64.** The GLM 5.3 Flash serverless host allocates every adapter slot at rank 64 and pads a smaller adapter with zeros, so a lower rank saves nothing at serving time.
- **Base vs prompted vs adapter arm.** An arm is one configuration evaluated on the same prompts: the untouched base model, the base model with an instruction (here a one-line ban on the tropes), or the base model with the trained adapter.
- **Held-out.** Prompts the model never trained on; every number in this notebook is measured on them.
- **Brief.** The task prompt in the Anti-slop pack: an article excerpt plus the instruction to write a tweet about it.
- **Detector / P(human).** The AI-text detector the pack was ranked with. It returns the probability that a text was written by a person; that probability is the gallery adapter's reward.
- **Tropes.** The habits people read as AI writing (em dashes, "not X but Y", stock phrases, emoji piles, rhetorical hooks), counted per 100 words by `antislop_tropes.py`. Kept out of the reward on purpose.
- **Openers.** The first three words of a post; "distinct openers of 40" counts how many different ones a set of 40 posts has.
- **Pass rate.** The share of outputs that meet a yes/no check (fits 280 characters, judged faithful).
- **Wilson interval.** A 95% confidence interval for a pass rate that stays inside 0-1 at small n.
- **Bootstrap interval / paired delta.** Resampling the per-post scores to get a 95% interval for a mean. A paired delta resamples the per-prompt difference between two arms, which is tighter than comparing two separate intervals.
- **nats.** The unit of a natural-log probability; a change of 1 nat multiplies a probability by e, about 2.7.
- **Datum.** The trainer's unit of input: one tokenized sequence plus a per-token weight saying which tokens the loss counts.
- **Scaffold.** The chat-template tokens around a response; on GLM 5.3 Flash the assistant turn opens with `<|assistant|><think></think>`. Weight 0, never trained.
- **Prefill.** Text placed at the start of the model's turn before it generates. `</think>` as prefill closes the think block so the model answers directly.
- **KL leash / beta.** DPO's `DPO_BETA` bounds how far the trained model may drift from the reference model, in log-probability units.
- **Bradley-Terry.** The model of a pairwise preference as a logistic function of a score difference; DPO's loss is that logistic regression on the margin.
- **Likelihood displacement.** The DPO failure where the preferred response gets *less* likely while the loss still improves, because the rejected response fell further. Section 5 shows it; the loop's early stop watches for it.
- **GRPO.** Group Relative Policy Optimization, the RL method the detailed guide (`GUIDE.md`)'s follow-up runs use: sample a group of answers per prompt, score them, push toward the above-average ones.
- **ORPO.** A DPO variant with an extra supervised term on the chosen response, which rules out displacement.
- **Checkpoint vs promoted model.** A checkpoint is a saved state inside the training session (`save_state`, also called DCP for distributed checkpoint, includes the optimizer state and is resumable); a promoted model is a checkpoint copied into the model catalog so it outlives the session.
- **Serverless vs dedicated deployment.** Serverless shares a pool and bills per token; a dedicated deployment is GPUs reserved for you, billed per hour. This notebook trains serverless and deploys nothing.
- **reasoning_effort.** How much a reasoning model thinks before answering. The pack's drafts were generated at low effort; the notebook generates with the think block closed by default.
- **swap_guard.** Asking the judge twice with the two answers in both orders and keeping the verdict only when they agree, which removes position bias.
- **kld/mean_k3.** The RL loop's measure of how far the sampler's log-probabilities drift from the trainer's; the same engine mismatch section 6 removes from DPO by scoring the reference on the trainer.


## See the result first

> **Recorded result.** Read from committed files under `results/`; no key, no network, nothing is run.

Before the mechanics, the outcome, rebuilt from the committed evaluation files in `results/` so it
needs no key and no network. Every arm wrote two tweets for each of the same 20 held-out briefs on one
deployment; the detector is the AI-text detector the pack was ranked with, and an "opener" is the
first three words of a post. Whiskers are 95% bootstrap intervals over the 40 posts.


In [ ]:
# Offline: reads results/*/samples.jsonl and draws the headline figure. No key, no network.
import sys
from pathlib import Path

# Resolve this folder from any cwd (Jupyter opened here, or Colab after the chdir in section 1).
HERE = Path.cwd()
ROOT = next((p for p in [HERE, *HERE.parents] if (p / "training" / "utils").is_dir()), None)
assert ROOT, f"run this notebook from inside the cookbook repo (cwd={HERE})"
STUDY = ROOT / "training" / "case-studies" / "dpo_antislop"
sys.path[:0] = [str(STUDY), str(STUDY.parent / "common")]

from antislop_figures import headline_figure   # per-arm bootstrap + the two-panel plot

fig, stats = headline_figure(STUDY / "results", save_to=STUDY / "figures" / "antislop_result.png")


**What the figure says.** RL step 20 is clear of the base model on the detector (interval 0.69 to 0.87 against 0.21 to 0.46), while DPO v1 and the prompted ban sit inside the base interval, so at 40 posts neither is distinguishable from base; the RL + diversity interval (0.54 to 0.79) overlaps RL step 20, so the detector cost of the diversity term is suggested, not established. The opener counts are counts, not estimates, and 4 of 40 is the finding.

**Read it as: the objective improved, the writing showed what the objective missed.** The reward went up and the writing got narrower. DPO alone did not
move the detector (0.33 to 0.30); 20 steps of GRPO on the same detector did (0.79) and cut the tropes
from 3.4 to 1.0 per 100 words with faithfulness intact, but 72% of its posts open with "New blog post:"
and the whole set shares four openers. Every post passes the judge on its own; a feed of them would read
as a bot. Penalizing openers shared within a sample group brought back 22 distinct openers of 40, and
the detector mean fell to 0.67; that interval overlaps RL step 20's, so whether the sameness carried
part of the score is suggested, not established, at 40 posts. The one-line prompt that bans the tropes
removes them completely (0.2 per 100 words) and keeps all the variety, at a lower detector score, which
is why a cheap control arm belongs in every table. The README has the full table; the rest of this
notebook is the loop that trained the DPO arm.


### A preference pair from the pack

> **Recorded data.** One row of `pack/antislop_pairs.jsonl`, the frozen pack the gallery adapter trained on.
> Both sides share the prompt; the detector ranked the chosen draft above the rejected one by at least 0.1.


In [ ]:
import json
from pathlib import Path

PACK = STUDY / "pack" / "antislop_pairs.jsonl"
pairs = [json.loads(line) for line in PACK.read_text().splitlines() if line.strip()]
pair = next(p for p in pairs if p["platform"] == "tweet")
print(f"{len(pairs)} pairs in the pack; showing {pair['brief_id']}\n")
print("PROMPT (user turn, first 300 chars):\n ", pair["chosen"]["messages"][-2]["content"][:300], "...\n")
print("CHOSEN (ranked more human by the detector):\n ", pair["chosen"]["messages"][-1]["content"], "\n")
print("REJECTED:\n ", pair["rejected"]["messages"][-1]["content"])


### Before and after, recorded

> **Recorded result.** Base `glm-5p3-flash` against the RL step 20 adapter (`glm53-antislop-rl`), from
> `results/2026-10-08/antislop-eval-rl/samples.jsonl`: 20 held-out briefs, 2 samples per arm, 40 posts per arm,
> all arms on one deployment. One brief, first sample of each arm, with the scores the evaluation recorded.


In [ ]:
RUN = STUDY / "results" / "2026-10-08" / "antislop-eval-rl"
recorded = [json.loads(line) for line in (RUN / "samples.jsonl").read_text().splitlines() if line.strip()]
summary = json.loads((RUN / "summary.json").read_text())
brief = [r for r in recorded if r["arm"] == "base"][2]["brief_id"]
print(f"brief: {brief}    (n = {summary['summary']['base']['tweet']['n']} posts per arm, evaluated 2026-10-08)\n")
for arm, label in (("base", "BASE  glm-5p3-flash"), ("rl_s20", "AFTER DPO + 20 GRPO steps  glm53-antislop-rl")):
    r = next(x for x in recorded if x["arm"] == arm and x["brief_id"] == brief)
    print(f"--- {label} ---")
    print(f"detector P(human) {r['detector']:.2f} | tropes/100w {r['tropes_per_100w']:.1f} | faithful {r['faithful']:.1f}")
    print(" ", r["text"], "\n")


## 1. Setup

Everything below is plain Python plus the repo's own renderer and judge helpers, so the setup is a
path and a key. Run this from inside the cookbook repo: the cells below import `training.*` and the shared
helpers in `case-studies/common/`. On Colab, uncomment the block at the top of the next cell first.
`FIREWORKS_API_KEY` is read here but only needed once `RUN_LIVE = True`.


In [ ]:
# --- Colab only: uncomment this block ------------------------------------
# Installs the repo's own pinned deps (transformers, torch, tinker, ...) from
# training/pyproject.toml rather than re-listing them here, so the two can't
# drift. Needs Python >= 3.11 (Colab's runtime is 3.12). If pip upgrades torch,
# restart the runtime once, then re-run from this cell.
# !git clone -q https://github.com/fw-ai/cookbook.git /content/cookbook
# %pip install -q -e /content/cookbook/training matplotlib openai
# import os; os.chdir("/content/cookbook/training")   # must chdir: ! cd does not persist
# -------------------------------------------------------------------------

import os, sys, json, time, math, random
from pathlib import Path

# Repo root on sys.path so `training.*` and case-studies/common/* import.
HERE = Path.cwd()
ROOT = next((p for p in [HERE, *HERE.parents] if (p / "training" / "utils").is_dir()), None)
assert ROOT, f"run this notebook from inside the cookbook repo (cwd={HERE})"
sys.path.insert(0, str(ROOT))
STUDY = ROOT / "training" / "case-studies" / "dpo_antislop"   # this folder, whatever the cwd
sys.path.insert(0, str(ROOT / "training" / "case-studies" / "common"))
sys.path.insert(0, str(STUDY))

try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / "training" / ".env")
except ImportError:
    pass

# Only the `%%live` cells need a key; a RUN_LIVE = False dry run works without one.
API_KEY = os.environ.get("FIREWORKS_API_KEY")
if not API_KEY:
    print("FIREWORKS_API_KEY is not set -- fine for a dry run; required once RUN_LIVE = True")
API_BASE = os.environ.get("FIREWORKS_BASE_URL", "https://api.fireworks.ai")
print("repo root:", ROOT)

# Spend gate. Cells that call the paid serverless trainer start with `%%live`
# and execute only when RUN_LIVE = True (config cell). Otherwise they print a
# skip notice, so Run-All on a fresh kernel costs nothing.
from IPython import get_ipython
from IPython.core.magic import register_cell_magic


@register_cell_magic
def live(line, cell):
    ns = get_ipython().user_ns
    if not ns.get("RUN_LIVE"):
        print("skipped: RUN_LIVE = False (this cell calls the paid serverless trainer)")
        return
    if not ns.get("API_KEY"):
        raise RuntimeError("RUN_LIVE = True needs FIREWORKS_API_KEY "
                           "(export FIREWORKS_API_KEY=fw_... or put it in training/.env)")
    exec(compile(cell, "<live cell>", "exec"), ns)

**The config cell.** Every knob of the run in one place: spend gate, model and renderer, LoRA shape, step
budget, judge, data path. Look at `RUN_LIVE` (False by default) and the `STEPS` / `MAX_PAIRS` comments,
which give the smoke-run and full-run values. The comments mention *likelihood displacement*; the glossary
defines it and section 5 shows it.


In [ ]:
# ----- CONFIG -------------------------------------------------------------
# False: Run-All spends nothing -- every `%%live` cell is skipped.
# True:  those cells call the paid serverless trainer. The defaults below are
#        a smoke run; the full-run values are in the comments and the README.
RUN_LIVE = False

# glm-5p3-flash is on the serverless TRAINING pool -- a much shorter list than
# the inference catalog. The tokenizer is the public Hugging Face one; no
# remote code, no gated download.
BASE_MODEL      = "accounts/fireworks/models/glm-5p3-flash"
TOKENIZER_MODEL = "zai-org/GLM-5.3-Flash"

# PINNED, not auto-resolved, so the name in use is the name in the log. For
# this tokenizer "" resolves to the same thing. "glm53_flash" writes an EMPTY
# <think></think> block before every training response, which is right for
# pairs that carry no reasoning (UltraFeedback, the Anti-slop pack). If your
# pairs include `reasoning_content`, use "glm53_flash_preserve_thinking" so the
# reasoning is trained too.
RENDERER_NAME = "glm53_flash"

# How sections 10-11 prompt the two models. False (default) ends the generation
# prompt with the same "<|assistant|><think></think>" scaffold the pairs were
# trained under, so both models answer directly and a 32-prompt win-rate takes
# minutes. True leaves the think block open, so both models reason at GLM's
# default maximum effort first: a fair look at the base model's native
# behavior, but 30-60 s per generation and an hour-plus win-rate.
GENERATE_WITH_REASONING = False

# Rank 64 is what the GLM 5.3 Flash serverless training pool and the shared
# multi-LoRA serving host are built around: every GPU slot is allocated at
# rank 64 and smaller adapters are zero-padded into it, so a lower rank saves
# nothing at serving time. Alpha 128 keeps the usual 2x alpha-to-rank scaling.
LORA_RANK   = 64
LORA_ALPHA  = 128
MAX_SEQ_LEN = 32768

# STEPS is a CEILING -- EARLY_STOP_CHOSEN_DROP is what ends the run. Aim for
# about one epoch: (MAX_PAIRS - EVAL_PAIRS) / BATCH_PAIRS.
STEPS       = 10           # smoke run. Full run: 370 (~1 epoch, 2.5-3 h)
BATCH_PAIRS = 8
LR          = 1e-5         # at rank 64 / alpha 128 this is a gentle setting; see the README
DPO_BETA    = 0.1          # the KL leash (section 5)
EVAL_PAIRS  = 32           # the headline number rests on these; 8 is too few
EVAL_INTERVAL = 20

# Stop when held-out `chosen` falls this many nats below baseline: that is
# likelihood displacement (section 5), and more steps only make it worse.
EARLY_STOP_CHOSEN_DROP = 2.0

# save_state = weights + optimizer state. Portable and resumable, but not
# promotable (that is save_weights_for_sampler). Aligned to EVAL_INTERVAL so
# every step the guard can name is a step you can return to.
DCP_SAVE_INTERVAL = EVAL_INTERVAL
assert not DCP_SAVE_INTERVAL or (EVAL_INTERVAL and DCP_SAVE_INTERVAL % EVAL_INTERVAL == 0), (
    "checkpoints are written at eval steps: make DCP_SAVE_INTERVAL a multiple of EVAL_INTERVAL")
DCP_TIMEOUT_S     = 900.0
RESUME_FROM = None         # "<account>/<run-id>/<checkpoint>" to continue a run

JUDGE_MODEL = "accounts/fireworks/models/kimi-k3"   # different family from the base: no self-bias
N_WINRATE   = EVAL_PAIRS   # win-rate runs on held-out prompts only; raise EVAL_PAIRS to grow it
assert N_WINRATE <= EVAL_PAIRS, "N_WINRATE cannot exceed the held-out set (EVAL_PAIRS)"

SEED = 0                   # demo reproducibility; comment out for real runs
random.seed(SEED)

DATA_DIR  = STUDY / "data"
# SEED picks which UltraFeedback rows are drawn, so it is part of the cache key:
# a new SEED downloads a new sample instead of reusing the old one.
DEMO_DATA = DATA_DIR / f"ultrafeedback_preferences_seed{SEED}.jsonl"
MAX_PAIRS = 200            # smoke run. Full run: 3000. UltraFeedback has ~63k
DATA_JSONL = None          # your own preference JSONL, e.g. the Anti-slop pack

## 2. The data: preference pairs

The data is the whole specification of what DPO will learn, so it is worth seeing one row before any
math. DPO's entire input is **pairs of answers to the same prompt, plus which one won**. No scores, no
rankings, no reward function.

By default we download [UltraFeedback binarized preferences](https://huggingface.co/datasets/argilla/ultrafeedback-binarized-preferences)
and convert it in this notebook, nothing else to run. It lands in a **local JSONL file** (path printed
below) so you can open it and see exactly what DPO consumes, and it's reused on re-runs rather than
re-downloaded.

To train on **your own** data instead, set `DATA_JSONL` in the config above; the validator below checks
it for you.

The download is network, not money, and happens once; the file is cached under `data/`. Without a
network (or with `HF_HUB_OFFLINE=1`) the cell says so and uses the committed Anti-slop pack instead,
so the rest of the notebook still runs.

**What the next cell does.** Downloads or reuses the demo JSONL and loads at most `MAX_PAIRS` rows.
Look at the row count and the path it prints; open the file to see exactly what DPO consumes.

In [ ]:
from training.examples.dpo.prepare_data import row_to_preference
from training.utils.data import normalize_preference_row


def download_ultrafeedback(dest, max_pairs, seed=0):
    '''Download UltraFeedback, convert to the chosen/rejected schema, write JSONL.

    The row conversion is the shared `row_to_preference` from
    examples/dpo/prepare_data.py: it skips empty and no-op pairs
    (identical responses carry zero signal).
    '''
    from datasets import load_dataset

    ds = load_dataset("argilla/ultrafeedback-binarized-preferences", split="train")
    idx = list(range(len(ds)))
    random.Random(seed).shuffle(idx)

    written = 0
    dest.parent.mkdir(parents=True, exist_ok=True)
    with dest.open("w") as f:
        for i in idx:
            if written >= max_pairs:
                break
            pref = row_to_preference(ds[i])
            if pref is None:
                continue
            line = json.dumps(pref, ensure_ascii=False)
            # Escape the line-breaking characters json.dumps leaves raw, so the
            # file is one record per line for ANY reader, not just ours.
            for ch in ("\u2028", "\u2029", "\u0085"):
                line = line.replace(ch, "\\u%04x" % ord(ch))
            f.write(line + "\n")
            written += 1
    return written

def read_jsonl(path):
    return [json.loads(l) for l in path.read_text().split("\n") if l.strip()]

def row_count(path):
    return sum(1 for l in path.read_text().split("\n") if l.strip())


data_path = Path(DATA_JSONL) if DATA_JSONL else DEMO_DATA

if DATA_JSONL:
    print(f"using your data: {data_path}")
else:
    # Re-download when the cached file is SMALLER than MAX_PAIRS. Checking only
    # for existence would silently train on a stale, smaller set after you raise
    # MAX_PAIRS -- a different experiment than the config claims.
    have = row_count(data_path) if data_path.exists() else 0
    if have >= MAX_PAIRS:
        print(f"reusing existing download ({have} rows >= MAX_PAIRS={MAX_PAIRS})")
    else:
        if have:
            print(f"cached file has {have} rows but MAX_PAIRS={MAX_PAIRS} -- re-downloading")
        print("downloading argilla/ultrafeedback-binarized-preferences ...")
        try:
            n = download_ultrafeedback(data_path, MAX_PAIRS, seed=SEED)
            print(f"wrote {n} pairs -> {data_path}")
        except Exception as e:   # offline, HF_HUB_OFFLINE=1, or the Hub is unreachable
            print(f"could not download UltraFeedback ({type(e).__name__}: {str(e)[:100]})")
            if have:
                print(f"continuing with the cached {have}-row file instead")
            else:
                data_path = STUDY / "pack" / "antislop_pairs.jsonl"
                print(f"no network needed to continue: using the committed Anti-slop pack -> {data_path}")

assert data_path.exists(), f"no data at {data_path}"
raw_rows = read_jsonl(data_path)

# MAX_PAIRS is authoritative: a file with extra rows is truncated, not silently
# used in full, so changing MAX_PAIRS always changes the experiment predictably.
if not DATA_JSONL and len(raw_rows) > MAX_PAIRS:
    print(f"file has {len(raw_rows)} rows; using the first MAX_PAIRS={MAX_PAIRS}")
    raw_rows = raw_rows[:MAX_PAIRS]

print(f"{len(raw_rows)} rows loaded ({data_path.stat().st_size / 1024:.0f} KB on disk)")
print(f"\nOpen it yourself:  head -c 800 {data_path}")

### Check the data before you spend anything on it

Three on-disk schemas are accepted (`training/utils/data.py`), all normalized to
`{"chosen": ..., "rejected": ...}`:

| Format | Shape |
| --- | --- |
| chosen / rejected | `{"chosen": {"messages": [...]}, "rejected": {"messages": [...]}}` |
| OpenAI preference | `{"input": {"messages": [...]}, "preferred_output": [...], "non_preferred_output": [...]}` |
| scored samples | `{"samples": [{"messages": [...], "evals": {"score": 1.0}}, {... "score": 0.0}]}` |

The validator checks schema *and* the two failures that produce **no error and a meaningless run**:
**prompt prefixes that differ** between chosen and rejected (the margin then measures the prompt, not
the response) and **identical responses** (zero gradient).

**What the next cell does.** Validates every row, then holds out `EVAL_PAIRS` rows by prompt group.
Look at `valid / total`, the formats seen, and the first rejected rows if any.

In [ ]:
from antislop_data import validate_preference_rows   # schema + the two silent failures; see antislop_data.py

rows, data_report = validate_preference_rows(raw_rows)
assert rows, "no valid preference rows -- fix the problems reported above before training"
assert len(rows) > EVAL_PAIRS, f"need more than EVAL_PAIRS={EVAL_PAIRS} valid rows, got {len(rows)}"

# Hold out about EVAL_PAIRS rows, by PROMPT GROUP, never by row. A pack can
# carry several pairs per prompt (the Anti-slop pack has up to 3 per brief);
# splitting rows would leave the held-out margin and win-rate scored on
# prompts the model trained on. Group key: `brief_id` when the pack has one,
# else the prompt text. Fixed membership keeps before/after comparable.
def _group_key(row):
    if row.get("brief_id"):
        return str(row["brief_id"])
    msgs = (normalize_preference_row(row) or {}).get("chosen", {}).get("messages") or []
    return json.dumps([(m.get("role"), m.get("content")) for m in msgs[:-1]])

eval_rows, train_rows, held = [], [], set()
for row in reversed(rows):                      # fill the held-out set from the end, whole groups at a time
    k = _group_key(row)
    if k in held or (len(eval_rows) < EVAL_PAIRS and k not in {_group_key(r) for r in train_rows}):
        eval_rows.append(row); held.add(k)
    else:
        train_rows.append(row)
train_rows.reverse(); eval_rows.reverse()
assert not ({_group_key(r) for r in train_rows} & held), "a prompt group leaked across the split"
print(f"\n{len(train_rows)} train / {len(eval_rows)} held out ({len(held)} prompt groups)")


In [ ]:
# Look at one pair before touching any math.
ex = normalize_preference_row(train_rows[0])
print("PROMPT:\n ", ex["chosen"]["messages"][0]["content"][:400], "\n")
print("CHOSEN (preferred):\n ", ex["chosen"]["messages"][-1]["content"][:400], "\n")
print("REJECTED:\n ", ex["rejected"]["messages"][-1]["content"][:400])

## 3. Connect to serverless training

DPO needs two models, one that moves and one that stays put, and serverless hands you both from a single
session. One service object hands you both a **training client** (weights that move) and **sampling clients**
(frozen snapshots). No trainer job, no deployment, no lifecycle to manage.

The **renderer** is the chat-template layer: it turns a message list into the exact token sequence this
base model was trained on. We pin it rather than auto-resolving, see the CONFIG comment.

The tokenizer and renderer are local and free, so they load even with `RUN_LIVE = False`; the
section 6 checks below use them. The tokenizer downloads once from Hugging Face (network, not money);
without a network the cell says so and the section 6 checks are skipped. Attaching to the trainer is the first `%%live` cell.

This attaches to a shared pool, so it needs no GPU quota. It can queue for a minute or two when the
pool is busy, and can fail outright if the pool is at capacity, just re-run the cell.

In [ ]:
import tinker
from fireworks.training.sdk import FiretitanServiceClient
from training.renderer import get_text_content
from training.renderer.tokenizer import get_tokenizer
from training.utils.losses import make_batch_dpo_loss_fn
from training.utils.supervised import (
    build_renderer,
    render_preference_pair,
    resolve_renderer_name,
)


def serverless_base_url(base_url: str) -> str:
    root = base_url.rstrip("/")
    if root.endswith("/training/v1/serverless"):
        return root
    if root.endswith("/training/v1"):
        return f"{root}/serverless"
    return f"{root}/training/v1/serverless"


try:
    tokenizer = get_tokenizer(TOKENIZER_MODEL)
except Exception as e:   # no network and no cached copy: the download is network, not money
    tokenizer = renderer = resolved_renderer = None
    print(f"could not load the {TOKENIZER_MODEL} tokenizer ({type(e).__name__}). It downloads once from")
    print("Hugging Face; until it loads, the rendering checks in section 6 print a skip notice. Re-run this cell online.")
else:
    # RENDERER_NAME = "" means "resolve from the tokenizer". Resolve it explicitly
    # too, so the name actually in use is printed rather than an empty string.
    resolved_renderer = RENDERER_NAME or resolve_renderer_name(TOKENIZER_MODEL)
    renderer = build_renderer(tokenizer, TOKENIZER_MODEL, resolved_renderer)
    print(f"renderer {resolved_renderer}"
          f"{' (auto-resolved)' if not RENDERER_NAME else ' (pinned)'}")


In [ ]:
%%live
service = FiretitanServiceClient(api_key=API_KEY, base_url=serverless_base_url(API_BASE))

if RESUME_FROM:
    # Restores weights AND Adam momentum. The reference then anchors at the
    # RESUMED weights, so margins are relative to the resume point. Keep
    # dataset, SEED and BATCH_PAIRS identical.
    print(f"resuming from {RESUME_FROM}")
    training_client = service.create_training_client_from_state_with_optimizer(RESUME_FROM)
else:
    training_client = service.create_lora_training_client(base_model=BASE_MODEL, rank=LORA_RANK, alpha=LORA_ALPHA)

print(f"attached to pooled trainer for {BASE_MODEL}")
print(f"  LoRA rank {LORA_RANK}, alpha {LORA_ALPHA}")
print(f"  training session: {service.training_session_name or service.training_session_id}")

## 4. The reference model

> **Live.** The next code cell starts with `%%live` and calls the paid serverless trainer. Its real output appears only with `RUN_LIVE = True`; by default it prints a skip notice.

The reference is what makes DPO's reward a *change* rather than an absolute, and it costs two lines here.
DPO needs a **frozen copy of the model you started from**, a "reference." Its only job is to answer:
*what did you think of this text before training?*

On serverless you get one in two lines. `save_weights_for_sampler` writes the current weights to a
snapshot; because the LoRA adapter is **zero-initialized** (LoRA's `B` matrix starts at zero, so the
adapter is a no-op until the first optimizer step), that snapshot *is* the untouched base model. Take
it **before any optimizer step** and it stays frozen for the whole run.

From here on two copies exist: the training client's weights (moving) and this snapshot (frozen).

One clarification about what this snapshot is *for*. DPO needs the reference's **logprobs**, and we get
those from the training client itself in section 6, before any optimizer step, same engine as the
policy, which matters more than it sounds like it should. This snapshot exists so we can later
**generate** from the untouched base model and read it side by side with the tuned one in sections 10-11.
Scoring and sampling, two different jobs.

In [ ]:
%%live
ref_path = training_client.save_weights_for_sampler("dpo-ref").result().path
assert ref_path, "save_weights_for_sampler returned no path"
reference_client = service.create_sampling_client(model_path=ref_path, tokenizer=tokenizer)
print("frozen reference:", ref_path)

## 5. What the reference buys you, and the trap it does *not* prevent (deep dive, skippable)

Skip to section 6 if you only want to run the experiment; nothing later depends on this section.
It is worked arithmetic and one figure, no GPU, and it is why the loop watches `chosen` and `rejected`
separately instead of the loss.

DPO turns each response's raw log-probability into an **implicit reward** measured against the reference:

$$r(y) = \beta\left[\log \pi_{\text{policy}}(y) - \log \pi_{\text{ref}}(y)\right]$$

$$\mathcal{L} = -\log \sigma\big(r(\text{chosen}) - r(\text{rejected})\big)$$

So "reward" means **how much more likely this text became relative to the base model** (in log units,
scaled by β), not absolute probability. The reference is the KL anchor from the RLHF objective that DPO
solves in closed form. That is what makes β a real leash: it bounds how far the policy may drift from
where it started.

`-log σ(·)` is logistic regression on the margin, the Bradley-Terry model of a pairwise preference
(glossary). It
is minimized by making the chosen response beat the rejected one, and it **saturates**, so a pair the
model already ranks correctly stops contributing gradient.

What the reference does **not** do is force the chosen response to become *more* likely. Look at row (b).

In [ ]:
sigmoid = lambda x: 1 / (1 + math.exp(-x))
REF = {"chosen": -10.0, "rejected": -20.0}   # the frozen reference's view of one pair

scenarios = {
    "start (policy == ref)":      (REF["chosen"], REF["rejected"]),
    "(a) learned it":             (-5.0,   -20.0),   # chosen went UP. what we want.
    "(b) displacement (typical)": (-14.0,  -40.0),   # BOTH went down, rejected further.
    "(c) collapse (extreme)":     (-100.0, -500.0),  # same shape, model destroyed.
}

hdr = f"{'scenario':28s} {'logp_chosen':>11s} {'logp_rej':>9s} {'r_chosen':>9s} {'r_rej':>8s} {'r_margin':>9s} {'loss':>8s}"
print(hdr); print("-" * len(hdr))
for name, (lp_c, lp_r) in scenarios.items():
    r_c = DPO_BETA * (lp_c - REF["chosen"])
    r_r = DPO_BETA * (lp_r - REF["rejected"])
    loss = -math.log(sigmoid(r_c - r_r))
    print(f"{name:28s} {lp_c:11.1f} {lp_r:9.1f} {r_c:+9.2f} {r_r:+8.2f} {r_c - r_r:+9.2f} {loss:8.4f}")

In [ ]:
# The same four scenarios on the loss curve. Offline, no GPU.
import matplotlib.pyplot as plt
from figures import COLORS, curve, save, style

style()
xs = [i / 10 for i in range(-30, 61)]
fig, ax = plt.subplots(figsize=(7.5, 3.8))
curve(ax, xs, [("DPO loss", [-math.log(sigmoid(x)) for x in xs])],
      title="Can the loss tell learning (a) from displacement (b)? Loss vs reward margin",
      xlabel="r(chosen) - r(rejected), beta-scaled", ylabel="-log sigmoid(margin)", marker="")
ax.axhline(math.log(2), color="gray", linestyle=":", linewidth=1)
ax.text(-2.9, math.log(2), "ln 2 = no preference", color="gray", va="bottom", fontsize=9)
offsets = {"start": (-30, 14), "(a)": (6, 12), "(b)": (6, 10), "(c)": (-150, 12)}
for name, (lp_c, lp_r) in scenarios.items():
    m = DPO_BETA * (lp_c - REF["chosen"]) - DPO_BETA * (lp_r - REF["rejected"])
    key = name[:3] if name.startswith("(") else "start"
    label = {"start": "start", "(a)": "(a) learned it", "(b)": "(b) displacement",
             "(c)": f"(c) collapse (margin {m:+.0f}, off the chart)"}[key]
    m_plot = min(m, 6)
    ax.plot([m_plot], [-math.log(sigmoid(m))], "o", color=COLORS["accent"], markersize=6, zorder=3)
    ax.annotate(label, (m_plot, -math.log(sigmoid(m))), xytext=offsets[key], textcoords="offset points", fontsize=9)
fig.tight_layout()
print("saved", save(fig, "figures/dpo_loss_vs_margin.png"))


**What the figure says.** Loss is a function of the margin alone, so (a) and (b) sit on the same curve and the curve is flat past a margin of about 3: there is no interval here, this is the exact formula.

Row **(b)** is the one to sit with: the loss is ~4x better than the start, and the chosen response
became **4x less likely** than the base model made it. The objective is satisfied only because the
rejected response fell further. Row (c) is the same shape taken to its conclusion.

This is **likelihood displacement**: the margin can rise while the chosen response's own likelihood
falls. The reference bounds drift in units of β; it does not forbid this.

**So watch `chosen_reward` and `rejected_reward` separately.** Healthy looks like:

```
chosen_reward   ↑  (or at least flat)
rejected_reward ↓
```

Both falling together means you are degrading the model while the loss curve improves. Remedies: lower
`LR`, raise `DPO_BETA`, or switch to **ORPO** ([`recipes/orpo_loop.py`](../../recipes/orpo_loop.py)),
whose extra `L_SFT(chosen)` term penalizes exactly this. `EARLY_STOP_CHOSEN_DROP` watches it for you.

> **Three things get called "margin."** `r_margin` above is β-scaled. The **trainer's** `margin` is the
> same quantity *without* β (~10x larger at β=0.1; same sign, same diagnostic). Section 8's **held-out
> margin** is a third thing: plain `log π(chosen) − log π(rejected)`, no reference, no β.

## 6. Render the pairs, then score them against the reference

The loss compares two whole token sequences, so before any step we need the pairs as tokens and the
reference's opinion of each, scored once and cached.

**Render.** `render_preference_pair` tokenizes both sides into **datums**, a tokenized sequence plus a
per-token weight for which tokens count. Prompt tokens get weight 0, response tokens 1: both sides
share the identical prompt, so training on it would be noise.

**Score with the reference.** We compute the frozen model's logprobs **once, up front, through the
training client's own `forward`**, before any `optim_step`, so the weights it sees *are* the step-0
reference weights.

Same engine matters: asking the reference *sampling* client instead works, but the sampling and
training engines return different logprobs for identical weights (different kernels, and for MoE
different expert routing). Measured, that put a **±2-3 noise floor on `margin` per pair**, larger than
the signal, which makes `margin`, `accuracy`, and both rewards unreadable. Scoring with the same engine
that later computes the policy's logprobs makes step 0 agree to exactly 0.0. RL loops track this same
divergence as `kld/mean_k3`; in DPO it lands inside the objective.

Cached by row index and never recomputed, the reference is frozen.

In [ ]:
def render_pair(row):
    norm = normalize_preference_row(row)
    if norm is None:
        raise ValueError("row does not match a supported preference schema")
    pair = render_preference_pair(
        norm["chosen"], norm["rejected"],
        renderer=renderer, tokenizer=tokenizer, max_seq_len=MAX_SEQ_LEN,
    )
    if pair is None:
        raise ValueError("pair failed to render (empty messages or over max_seq_len)")
    return pair


def interleave(pairs):
    '''[chosen_0, rejected_0, chosen_1, rejected_1, ...] -- the layout the loss pairs by index.'''
    return [d for p in pairs for d in (p.chosen_datum, p.rejected_datum)]

Schema validity isn't the whole story: a pair also has to **render and fit**. This is the second half
of the data check, and it needs the tokenizer, which is why it lives here rather than in section 2.
`render_preference_pair` returns `None` when a pair exceeds `MAX_SEQ_LEN`, so we drop those now rather
than crashing mid-training.

In [ ]:
def validate_rendering(rows, label):
    '''Drop pairs that fail to render or exceed MAX_SEQ_LEN; report token stats.'''
    kept, dropped, lengths = [], [], []
    for i, row in enumerate(rows):
        try:
            pair = render_pair(row)
        except ValueError as e:
            dropped.append((i, str(e)))
            continue
        lengths += [pair.chosen_datum.model_input.length,
                    pair.rejected_datum.model_input.length]
        kept.append(row)

    if lengths:
        lengths.sort()
        print(f"{label}: {len(kept)}/{len(rows)} pairs render | "
              f"tokens min {lengths[0]} / median {lengths[len(lengths) // 2]} / max {lengths[-1]} "
              f"(MAX_SEQ_LEN {MAX_SEQ_LEN})")
    else:
        print(f"{label}: 0/{len(rows)} pairs render")
    for i, msg in dropped[:3]:
        print(f"  dropped row {i}: {msg}")
    return kept


if renderer is None:
    print("skipped: tokenizer not loaded (section 3); re-run that cell online, then this one")
else:
    train_rows = validate_rendering(train_rows, "train")
    eval_rows = validate_rendering(eval_rows, "held out")
    assert train_rows and eval_rows, "no renderable pairs left -- raise MAX_SEQ_LEN or check the renderer"

### The training boundary (deep dive, skippable)

Don't take that on faith, check it on **your** data. This prints the assistant-turn scaffold your
renderer emits and the exact span the DPO loss compares. If the compared span doesn't start at your
response text, your renderer doesn't match your data.

On GLM 5.3 Flash expect the scaffold to end in `<|assistant|><think></think>` and the compared span
to start right after it, at the first word of the response. The empty think block belongs to the prompt
side, weight 0, so DPO trains the answer and nothing about reasoning. Section 10 shows what that means
at generation time.

In [ ]:
def show_training_boundary(row):
    pair = render_pair(row)
    rs = pair.response_start
    c = pair.chosen_datum.model_input.to_ints()
    j = pair.rejected_datum.model_input.to_ints()

    print(f"response_start = {rs}   chosen len = {len(c)}   rejected len = {len(j)}")
    print(f"prefixes identical up to response_start: {c[:rs] == j[:rs]}")
    print()
    print("assistant-turn scaffold the renderer emits (the last 12 prompt tokens):")
    print("   ", repr(tokenizer.decode(c[max(0, rs - 12):rs])))
    print()
    print(f"what the DPO loss actually compares (from index rs-1 = {rs - 1}):")
    print("    chosen  :", repr(tokenizer.decode(c[rs - 1:rs + 14])))
    print("    rejected:", repr(tokenizer.decode(j[rs - 1:rs + 14])))


if renderer is None:
    print("skipped: tokenizer not loaded (section 3)")
else:
    show_training_boundary(train_rows[0])

> **Live.** The next code cell starts with `%%live` and calls the paid serverless trainer. Its real output appears only with `RUN_LIVE = True`; by default it prints a skip notice.

Now the reference pass. **This must run before the first `optim_step`**, that is what makes these the
step-0 weights. It is the one upfront cost in the notebook; everything after it is cached.

**What the next cell does.** Fixes the training order, then scores every batch the loop will visit with
the step-0 weights and caches the result. Look at the epoch count it derives and the number of batches cached.

In [ ]:
%%live
REF_CACHE = {}


def precompute_reference_logprobs(batches, timeout=900):
    '''Reference logprobs for every row, from the TRAINING client's forward pass.

    Must be called before any optim_step: the weights the forward sees are then
    still the step-0 (zero-init LoRA = base model) reference weights.

    Scores the loop's own batches, not a re-chunked list of rows. Trainer
    logprobs are batch-size dependent (measured at up to ~2.8 nats on a single
    token), so every reference forward is a full BATCH_PAIRS batch -- and within
    the first epoch it is the very batch dpo_step will later score. A short
    trailing chunk would reintroduce the noise this cell exists to remove.

    Returns nothing; fills REF_CACHE keyed by dataset row index. The arrays come
    back already in the datum layout the DPO loss slices, so no realignment is
    needed -- unlike a sampling-client compute_logprobs, which prepends a None.
    '''
    n_forwards = 0
    for batch in batches:
        # Checked per batch, not up front: once the first epoch has filled the
        # cache, later-epoch batches need no forward at all.
        if all(i in REF_CACHE for i in batch):
            continue
        pairs = [render_pair(train_rows[i]) for i in batch]
        fwd = training_client.forward(interleave(pairs), "cross_entropy").result(timeout=timeout)
        n_forwards += 1
        for k, idx in enumerate(batch):
            if idx in REF_CACHE:       # first score wins; the reference never changes
                continue
            REF_CACHE[idx] = (
                [float(x) for x in fwd.loss_fn_outputs[2 * k]["logprobs"].data],
                [float(x) for x in fwd.loss_fn_outputs[2 * k + 1]["logprobs"].data],
            )
        print(f"  reference logprobs: {len(REF_CACHE)} pairs cached", end="\r", flush=True)
    print(f"  reference logprobs: {len(REF_CACHE)} pairs cached from {n_forwards} batches")


# Fix the training order HERE so we score exactly the rows the loop visits.
# REF_CACHE is keyed by index into train_rows, so train_rows must be FINAL --
# re-running the data cells with a different config would shift every index.
# Snapshot the rows themselves (not just the count) so the loop can verify them.
REF_TRAIN_ROWS = tuple(train_rows)


def resume_start_step(ref):
    '''Optimizer steps already taken by a `.../dpo-NNNN` checkpoint (saved after
    step index NNNN, so NNNN + 1 updates).'''
    suffix = ref.rstrip("/").split("/")[-1].rsplit("-", 1)[-1]
    assert suffix.isdigit(), f"RESUME_FROM must name a dpo-NNNN checkpoint, got {ref!r}"
    return int(suffix) + 1


# A resumed run continues where the checkpoint stopped: same data order, next
# batch, absolute step numbers -- instead of replaying the first batches.
START_STEP = resume_start_step(RESUME_FROM) if RESUME_FROM else 0
assert START_STEP < STEPS, f"checkpoint is already at step {START_STEP}; raise STEPS to continue"

order = list(range(len(train_rows)))
random.Random(SEED).shuffle(order)
# batches[k] is what optimizer step START_STEP + k trains on.
batches = [[order[(s * BATCH_PAIRS + i) % len(order)] for i in range(BATCH_PAIRS)]
           for s in range(START_STEP, STEPS)]
visited = [i for b in batches for i in b]
needed = list(dict.fromkeys(visited))          # de-duped, order preserved

# Step-driven, unlike recipes/dpo_loop.py (`epochs`). Print the derived
# epoch count rather than make you divide.
epochs = len(visited) / len(train_rows)
if START_STEP:
    print(f"resuming at step {START_STEP}")
print(f"{STEPS - START_STEP} steps x {BATCH_PAIRS} pairs = {len(visited)} pair-visits "
      f"over {len(needed)} distinct rows")
print(f"  = {epochs:.2f} epochs of the {len(train_rows)}-row train pool "
      f"({len(train_rows) - len(needed)} rows unused)"
      if epochs <= 1 else
      f"  = {epochs:.2f} epochs of the {len(train_rows)}-row train pool "
      f"(each pair seen ~{len(visited) / len(needed):.1f}x)")
print(f"  one full epoch would be STEPS = {len(train_rows) // BATCH_PAIRS}")
t0 = time.time()
precompute_reference_logprobs(batches)
print(f"done in {time.time() - t0:.1f}s")

> **Live.** The next code cell starts with `%%live` and calls the paid serverless trainer. Its real output appears only with `RUN_LIVE = True`; by default it prints a skip notice.

Now **prove the reference is what we claimed**: same weights, same engine, so the policy's logprobs at
step 0 must equal the cached reference logprobs *exactly*. Any nonzero value here means either an
optimizer step already ran, or the reference came from a different engine.

In [ ]:
%%live
# Compare using a FULL BATCH_PAIRS batch -- the same shape the reference pass
# and the training loop use. Scoring a single pair here would change the batch
# size and disagree by nats even though nothing is wrong.
_idx = list(REF_CACHE)[:BATCH_PAIRS]
_pairs = [render_pair(train_rows[i]) for i in _idx]
_pol = training_client.forward(interleave(_pairs), "cross_entropy").result()

_delta = 0.0
for _k, _i in enumerate(_idx):
    for _side in (0, 1):
        _ref = REF_CACHE[_i][_side]
        _cur = [float(x) for x in _pol.loss_fn_outputs[2 * _k + _side]["logprobs"].data]
        _delta = max([_delta] + [abs(a - b) for a, b in zip(_ref, _cur)])

print(f"max |policy - reference| logprob at step 0: {_delta:.2e}   (must be 0.0)")
print("If this is nonzero, either an optim_step already ran, or the reference")
print("was scored with a different batch size / engine than the policy.")

### Resumable checkpoints (deep dive, skippable)

`save_state` writes weights **plus optimizer state**, so a resume is a true continuation rather than a
warm start from cold momentum. It is a different artifact from the sampler snapshots sections 10-12 use;
see the config comment for the split.

The reference it prints is **portable**: unlike a sampler snapshot, it outlives this session and can be
loaded by a fresh kernel.

In [ ]:
saved_checkpoints = []


def dcp_reference(name):
    '''The portable `<account>/<run-id>/<checkpoint>` reference for RESUME_FROM.'''
    session = getattr(service, "training_session_name", None) or ""
    account = session.split("/")[1] if session.startswith("accounts/") else None
    run_id = getattr(training_client, "run_id", None)
    return f"{account}/{run_id}/{name}" if account and run_id else name


def save_dcp(step):
    '''Full trainer state -- what RESUME_FROM consumes. The name must end in a
    zero-padded step number; the resume path parses the step back out of it.
    '''
    name = f"dpo-{step:04d}"
    t0 = time.time()
    training_client.save_state(name).result(timeout=DCP_TIMEOUT_S)
    ref = dcp_reference(name)
    saved_checkpoints.append((step, ref))
    print(f"     checkpoint {name} ({time.time() - t0:.0f}s) -> {ref}")
    return ref

## 7. The training step

Everything above assembles into one function, the whole algorithm. Note what is **absent** compared to an RL loop: no
sampling, no reward function, no exploration, no group statistics. Just render, score against the
reference, and update.

One API detail: this calls **`forward_backward_custom`**, not `forward_backward`. A built-in loss (like
RL's `"importance_sampling"`) works on per-token arrays inside a single datum. DPO compares *two whole
sequences*, so the loss has to be a function you pass in, closed over the frozen reference arrays.

In [ ]:
adam = tinker.AdamParams(learning_rate=LR, beta1=0.9, beta2=0.95, eps=1e-12, weight_decay=0.0)


def dpo_step(indexed_batch):
    pairs = [(i, render_pair(r)) for i, r in indexed_batch]
    rendered = [p for _, p in pairs]

    loss_fn = make_batch_dpo_loss_fn(
        [REF_CACHE[i][0] for i, _ in pairs],    # reference chosen logprobs (cached at init)
        [REF_CACHE[i][1] for i, _ in pairs],    # reference rejected logprobs
        [p.response_start for p in rendered],   # where each response begins
        DPO_BETA,
    )
    fb = training_client.forward_backward_custom(interleave(rendered), loss_fn).result()
    training_client.optim_step(adam).result()

    m = getattr(fb, "metrics", None) or {}
    return {k: m.get(k) for k in
            ("dpo_loss", "margin", "accuracy", "chosen_reward", "rejected_reward")}

## 8. Held-out margin: the baseline

> **Live.** The next code cell starts with `%%live` and calls the paid serverless trainer. Its real output appears only with `RUN_LIVE = True`; by default it prints a skip notice.

The training log is noisy and measured on moving batches; a fixed held-out set measured the same way
every time is the curve to trust. A plain `forward` on the fixed held-out pairs, no backward, no optimizer. It returns **three** numbers,
because the margin alone is a *gap*: a gap widens identically whether chosen rose or both fell with
rejected falling faster. That is the section 5 blind spot, so we track chosen and rejected separately.

**The baseline value is not the result, the delta is.** Its *sign* is not meaningful either: this sums
logprobs, and longer text sums to a more negative number. On this dataset chosen responses average 277
tokens against rejected's 185, worth roughly 50 nats of margin from length alone. Length cancels in the
delta. (The DPO loss sums the same way, which is why DPO tends to make models more verbose.)

In [ ]:
%%live
def held_out_margin(pairs_per_forward=None):
    """Mean held-out (chosen, rejected, margin) summed logprobs.

    All three, not just the margin: a gap widens identically whether chosen rose
    or both fell with rejected falling faster (section 5). Chunked at
    BATCH_PAIRS per forward, same reason as the reference pass.
    """
    pairs_per_forward = pairs_per_forward or BATCH_PAIRS
    pairs = [render_pair(r) for r in eval_rows]
    tot_c = tot_r = 0.0
    for s in range(0, len(pairs), pairs_per_forward):
        chunk = pairs[s:s + pairs_per_forward]
        fwd = training_client.forward(interleave(chunk), "cross_entropy").result()
        for k, pair in enumerate(chunk):
            lp = max(0, pair.response_start - 1)
            tot_c += sum(fwd.loss_fn_outputs[2 * k]["logprobs"].data[lp:])
            tot_r += sum(fwd.loss_fn_outputs[2 * k + 1]["logprobs"].data[lp:])
    n = len(pairs)
    return {"chosen": tot_c / n, "rejected": tot_r / n, "margin": (tot_c - tot_r) / n}


def print_held_out(label, m, base=None):
    if base is None:
        print(f"{label}:  chosen {m['chosen']:+9.3f} | rejected {m['rejected']:+9.3f} "
              f"| margin {m['margin']:+9.3f}")
    else:
        print(f"{label}:  chosen {m['chosen']:+9.3f} ({m['chosen'] - base['chosen']:+7.3f}) "
              f"| rejected {m['rejected']:+9.3f} ({m['rejected'] - base['rejected']:+7.3f}) "
              f"| margin {m['margin']:+9.3f} ({m['margin'] - base['margin']:+7.3f})")


before = held_out_margin()
eval_history = [{"step": START_STEP, **before}]
print_held_out(f"held-out @ step {START_STEP:3d}", before)

## 9. Run the loop

> **Live.** The next code cell starts with `%%live` and calls the paid serverless trainer. Its real output appears only with `RUN_LIVE = True`; by default it prints a skip notice.

Now the loop itself, with the held-out check and the displacement guard built in so a run that is
quietly going wrong stops on its own.

`STEPS` is a ceiling; `EARLY_STOP_CHOSEN_DROP` is what usually ends the run.

Two reading notes:

- **`accuracy` and `margin` start at exactly 0.0.** Accuracy counts pairs whose *reference-adjusted*
  margin is strictly `> 0`. At step 0 the policy **is** the reference and both were scored by the same
  engine, so every margin is exactly zero. Nonzero here means the reference was scored by a different
  engine, see section 6.
- Per-step `margin` has a standard deviation around 0.7 purely from batch composition, so read trends
  over ten-plus steps. The held-out lines are what settle it.

> **`chosen_r` is the diagnostic, not the loss.** Divide it by `DPO_BETA` for nats moved against base.
>
> | symptom | reading |
> | --- | --- |
> | loss pinned at `ln 2`, `chosen_r` ±0.03 | not learning, raise `LR` or `LORA_RANK` |
> | `rej_r` falling, `chosen_r` flat or rising | healthy |
> | both falling, `chosen_r` diving | **likelihood displacement**, lower `LR`, raise `DPO_BETA`, or use ORPO |
>
> The trap: a displaced run has the *best* loss curve, the *best* accuracy, and the *largest* margin of
> any configuration. Every headline number says it won. Section 5 is about why.

**What the next cell does.** One `dpo_step` per batch, a held-out check every `EVAL_INTERVAL`, a checkpoint
at the same cadence, and the displacement guard. Look at the held-out lines: margin up with chosen not
falling is the healthy pattern; an EARLY STOP line names the best step to return to.

In [ ]:
%%live
# `order` was fixed in section 6 so the reference pass scored exactly these
# rows; re-shuffling here would visit rows with no cached reference logprobs.
assert tuple(train_rows) == REF_TRAIN_ROWS, (
    "train_rows changed after the reference pass -- REF_CACHE indices are stale. "
    "Re-run the reference-pass cell in section 6."
)
history = []
best_eval = {"step": None, "gain": float("-inf"), "drop": 0.0}
stopped_early = False
fmt = lambda v, s="+.4f": "n/a" if v is None else format(v, s)

print(f"{'step':>4s} {'loss':>8s} {'margin':>9s} {'acc':>7s} {'chosen_r':>9s} {'rej_r':>9s} {'secs':>6s}")
for step in range(START_STEP, STEPS):
    idx = batches[step - START_STEP]
    t0 = time.time()
    rec = dpo_step([(i, train_rows[i]) for i in idx])
    rec["step"], rec["wall"] = step, time.time() - t0
    history.append(rec)
    print(f"{step:4d} {fmt(rec['dpo_loss'], '.4f'):>8s} {fmt(rec['margin']):>9s} "
          f"{fmt(rec['accuracy'], '.1%'):>7s} {fmt(rec['chosen_reward']):>9s} "
          f"{fmt(rec['rejected_reward']):>9s} {rec['wall']:6.1f}")

    # Same fixed pairs every time, so this is the curve to trust.
    if EVAL_INTERVAL and (step + 1) % EVAL_INTERVAL == 0 or step == STEPS - 1:
        ev = {"step": step, **held_out_margin()}
        eval_history.append(ev)
        print_held_out(f"     held-out @ step {step:3d}", ev, before)

        if DCP_SAVE_INTERVAL and ((step + 1) % DCP_SAVE_INTERVAL == 0 or step == STEPS - 1):
            save_dcp(step)

        # Best point = largest margin gain while chosen has not fallen more
        # than half the stop threshold. (Held-out chosen wobbles ~0.1 nats, so
        # requiring drop <= 0 rejects every good step.)
        gain = ev["margin"] - before["margin"]
        drop = before["chosen"] - ev["chosen"]          # positive = chosen fell
        if drop <= (EARLY_STOP_CHOSEN_DROP or 2.0) / 2 and gain > best_eval["gain"]:
            best_eval = {"step": step, "gain": gain, "drop": drop}

        # Early stop on displacement. The margin will keep climbing past this
        # point; the model will keep getting worse at the preferred answers.
        if EARLY_STOP_CHOSEN_DROP and drop > EARLY_STOP_CHOSEN_DROP:
            print()
            print(f"  EARLY STOP at step {step}: held-out chosen fell {drop:.2f} nats "
                  f"(limit {EARLY_STOP_CHOSEN_DROP}).")
            print(f"  This is likelihood displacement -- margin is up {gain:+.2f} but the")
            print(f"  preferred responses became LESS likely. Running longer makes it worse.")
            if best_eval["step"] is not None:
                print(f"  Best point seen: step {best_eval['step']} "
                      f"(margin {best_eval['gain']:+.2f}, chosen {-best_eval['drop']:+.2f}).")
                print(f"  Set STEPS = {best_eval['step'] + 1} to stop there, or lower LR / raise DPO_BETA.")
                _ref = next((r for st, r in saved_checkpoints if st == best_eval["step"]), None)
                if _ref:
                    print(f"  Resume from that point with  RESUME_FROM = \"{_ref}\"")
            stopped_early = True
            break

# The last eval entry IS the current state -- recomputing it would waste an
# eval and, on an early stop, label it STEPS-1, a step that never ran.
after = eval_history[-1]
if not stopped_early and after["step"] != STEPS - 1:
    after = {"step": STEPS - 1, **held_out_margin()}
    eval_history.append(after)
    print()
    print_held_out(f"held-out @ step {STEPS - 1:3d}", after, before)

d_c = after["chosen"] - before["chosen"]
d_r = after["rejected"] - before["rejected"]
d_m = after["margin"] - before["margin"]

print()
print(f"held-out margin: {before['margin']:+.4f} -> {after['margin']:+.4f}  (delta {d_m:+.4f})")
if d_m > 0 and d_c < 0:
    share = abs(d_c) / (abs(d_c) + abs(d_r)) if (d_c or d_r) else 0
    print(f"\n  !! LIKELIHOOD DISPLACEMENT: the margin rose {d_m:+.2f} but the CHOSEN")
    print(f"     response became {abs(d_c):.2f} nats LESS likely. The gain came from the")
    print(f"     rejected response collapsing -- chosen is only {share:.0%} of the movement.")
    print(f"     See section 5: lower LR, raise DPO_BETA, or switch to ORPO.")
elif d_m > 0:
    print(f"\n  healthy: margin up {d_m:+.2f} with chosen {d_c:+.2f} (not falling).")
else:
    print(f"\n  margin did not improve ({d_m:+.2f}). See the troubleshooting note above.")

if stopped_early:
    _best_ref = next((r for st, r in saved_checkpoints if st == best_eval["step"]), None)
    print(f"\n  NOTE: these weights are from step {after['step']}, which is past the best")
    print(f"        point. The loop cannot rewind -- but the checkpoints can.")
    if _best_ref:
        print(f"        Recover the best weights in a fresh kernel with:")
        print(f"          RESUME_FROM = \"{_best_ref}\"")
    elif saved_checkpoints:
        print(f"        Nearest saved checkpoint: {saved_checkpoints[-1][1]}")
    else:
        print(f"        No checkpoints were saved (DCP_SAVE_INTERVAL = 0), so the only")
        print(f"        way back is a re-run with a smaller STEPS.")
elif best_eval["step"] is not None and best_eval["step"] < STEPS - 1 and d_c < -0.5:
    print(f"\n  NOTE: the best held-out point was step {best_eval['step']} "
          f"(margin {best_eval['gain']:+.2f}, chosen {-best_eval['drop']:+.2f}); "
          f"the run continued past it.")
    _ref = next((r for st, r in saved_checkpoints if st == best_eval["step"]), None)
    if _ref:
        print(f"        these weights are from step {after['step']}. To get the best ones,")
        print(f"        set RESUME_FROM = \"{_ref}\" in a fresh kernel.")

if saved_checkpoints:
    print(f"\nresumable checkpoints ({len(saved_checkpoints)}):")
    for st, ref in saved_checkpoints[-5:]:
        print(f"  step {st:4d}  {ref}")
    if len(saved_checkpoints) > 5:
        print(f"  ... and {len(saved_checkpoints) - 5} earlier")

> **Live.** The next code cell starts with `%%live` and calls the paid serverless trainer. Its real output appears only with `RUN_LIVE = True`; by default it prints a skip notice.

**What the next cell does.** Four training curves: loss, train margin and accuracy, the two implicit
rewards, and the held-out change from step 0. The last panel is the one to trust.

In [ ]:
%%live
from antislop_figures import training_curves   # the four panels; see antislop_figures.py

training_curves(history, eval_history)


## 10. Read the actual text

> **Live.** The next code cell starts with `%%live` and calls the paid serverless trainer. Its real output appears only with `RUN_LIVE = True`; by default it prints a skip notice.

Numbers say the margin moved; only the text says whether the model got better. Save the tuned weights as a sampler snapshot and generate from the frozen reference and the tuned model
on the **same held-out prompts**.

One thing to keep straight. Section 6 showed that every training response sat behind an
`<|assistant|><think></think>` scaffold: an empty, closed think block. The default here
(`GENERATE_WITH_REASONING = False`) generates under that same scaffold by prefilling `</think>`, so both
models answer directly and you are reading the distribution DPO actually trained. That is also what
makes the win-rate in section 11 take minutes rather than an hour.

Flip it to `True` and the prompt ends in an open `<think>`: both models reason at GLM 5.3 Flash's
default maximum effort before answering, `get_text_content` strips the reasoning, and you see how the
adapter behaves when the model is allowed to think first. Training never touched the think block, so
the adapter changes the answer either way; it does not change whether the model reasons. If you want
an adapter that skips reasoning on its own, that comes from training data (SFT on direct answers, as
in the [Caveman notebook](../sft_caveman/caveman_sft.ipynb)), and pairs that carry `reasoning_content` with the
`glm53_flash_preserve_thinking` renderer train the reasoning itself.

**What the next cell does.** Snapshots the tuned weights, warms up both samplers, then prints base and tuned
answers for two held-out prompts. Look for answers that are longer but not better; DPO sums logprobs.

In [ ]:
%%live
final_path = training_client.save_weights_for_sampler("dpo-final").result().path
tuned_client = service.create_sampling_client(model_path=final_path, tokenizer=tokenizer)
print("tuned snapshot:", final_path)

# Template markers that must never appear in a decoded answer. If they do, we
# are looking at raw scaffolding rather than content.
_TEMPLATE_MARKERS = ("<|assistant|>", "<|user|>", "<|system|>", "<|observation|>", "<think>")

# DPO pushes toward longer output (the loss SUMS logprobs, and this dataset's
# chosen responses average 277 tokens against rejected's 185), so the tuned
# model needs more headroom than the base model did. With
# GENERATE_WITH_REASONING = True the think tokens count against this too, so
# raise it to 4096 or more.
MAX_GEN_TOKENS = 2048 if not GENERATE_WITH_REASONING else 4096


def prompt_messages(row):
    """Everything before the final assistant turn. None if the row isn't that shape."""
    msgs = (normalize_preference_row(row) or {}).get("chosen", {}).get("messages") or []
    if not msgs or msgs[-1].get("role") != "assistant":
        return None
    return msgs[:-1] or None


def generate(client, messages, max_tokens=None, temperature=0.7, timeout=420):
    """Sample one response. Returns (text, status); status "ok" or a skip reason.

    Validates the visible content rather than trusting parser flags: `get_text_content`
    drops the <think> block, so an answer that was all reasoning comes back empty
    and is reported as such instead of being judged.
    """
    max_tokens = max_tokens or MAX_GEN_TOKENS
    # prefill="</think>" closes the think block in the prompt, which is exactly
    # the scaffold render_preference_pair trained on (section 6). Without it the
    # prompt ends in "<think>" and the model reasons before answering.
    prompt = renderer.build_generation_prompt(
        messages, prefill=None if GENERATE_WITH_REASONING else "</think>")
    params = tinker.SamplingParams(max_tokens=max_tokens, temperature=temperature,
                                   stop=renderer.get_stop_sequences())
    try:
        resp = client.sample(prompt=prompt, num_samples=1,
                             sampling_params=params).result(timeout=timeout)
    except Exception as e:
        # 503 while the pool loads an adapter. Skip one prompt rather than
        # abandoning the whole win-rate run.
        return "", f"sample failed ({type(e).__name__})"

    tokens = list(resp.sequences[0].tokens)
    text = get_text_content(renderer.parse_response(tokens)[0]).strip()
    if not text:
        return "", "empty"
    if len(tokens) >= max_tokens:
        return "", f"truncated at {max_tokens} tokens"
    if any(m in text for m in _TEMPLATE_MARKERS):
        return "", "template markers in output (renderer/prompt mismatch)"
    return text, "ok"


def warm_up(client, label, tries=8, wait=20):
    """Prime a pooled sampler: the pool 503s while loading an adapter on first
    use, which can exhaust the SDK's retries. A 16-token probe is enough -- it
    reports "truncated", which still proves the adapter is serving.
    """
    for i in range(1, tries + 1):
        _, st = generate(client, [{"role": "user", "content": "Say OK."}],
                         max_tokens=16, timeout=120)
        if st == "ok" or st.startswith("truncated"):
            print(f"  {label:24s} ready")
            return True
        print(f"  {label:24s} not ready ({st}) [{i}/{tries}] -- waiting {wait}s")
        if i < tries:
            time.sleep(wait)
    print(f"  {label:24s} STILL UNAVAILABLE -- the pool may be at capacity. Re-run this cell.")
    return False


print("warming up the samplers (the pool loads each adapter on first use):")
ready_base = warm_up(reference_client, "base (reference)")
ready_tuned = warm_up(tuned_client, "tuned")
if not (ready_base and ready_tuned):
    print("\n  One or both samplers are not serving yet. Re-run this cell before section 11;")
    print("  starting the win-rate now would skip every prompt.")
print()

for row in eval_rows[:2]:
    msgs = prompt_messages(row)
    if not msgs:
        continue
    print("=" * 100)
    print("PROMPT:", str(msgs[-1].get("content", ""))[:300])
    for label, client in (("BASE (frozen reference)", reference_client), ("AFTER DPO", tuned_client)):
        text, status = generate(client, msgs)
        print(f"\n--- {label} ---")
        print(f" {text[:700]}" if status == "ok" else f" (unusable: {status})")
    print()

## 11. Win-rate: the number you show a customer

> **Live.** The next code cell starts with `%%live` and calls the paid serverless trainer. Its real output appears only with `RUN_LIVE = True`; by default it prints a skip notice.

Margin is the honest training signal, but it isn't persuasive on a slide. So we ask an **impartial judge
model** to compare base and tuned answers on the same held-out prompts and report how often the tuned
model wins.

Four things make this a fair test rather than a flattering one:

- **A different model family judges** (`kimi-k3` scoring `glm-5p3-flash`), so there's no self-preference bias.
- **`swap_guard=True`** runs both orderings and keeps a verdict only when they agree after un-swapping,
  which removes position bias. Disagreements become ties.
- **Held-out prompts only**, never trained on.
- **And a caveat stated before you see the chart:** a few dozen prompts (`EVAL_PAIRS`) with a swap guard typically leaves only a
  handful of *decisive* verdicts. Treat this as a smoke test of the measurement, not an effect size. The
  number you'd publish comes from 200+ prompts.

No deployment needed: we generate from the two snapshots we already have and only call the API for the
judge. `pairwise_judge` swallows judge errors as `"tie"`, so a bad `JUDGE_MODEL` id would look like a
tied model. The cell makes one direct judge call first so that case fails loudly instead.

**What the next cell does.** Generates both arms on every held-out prompt and asks the judge twice per pair.
Look at the decisive count and the per-side skips; an asymmetric skip count biases the win-rate.

In [ ]:
%%live
from eval_common import fw_client, pairwise_judge


def _norm_verdict(v):
    """Collapse a judge verdict to exactly "A", "B", or "tie"."""
    v = str(v).strip().upper()
    return v if v in ("A", "B") else "tie"


def judge_pair(judge, model, messages, a, b):
    """Swap-guarded pairwise verdict: "A", "B", or "tie".

    Both orderings; a decisive verdict only when they agree after un-swapping,
    so position bias becomes a tie. Same two judge calls as swap_guard=True --
    but the shared helper's remap KeyErrors when a judge returns "Tie" (it
    compares lowercase then applies .upper()), so we normalize and un-swap here.
    """
    v1 = _norm_verdict(pairwise_judge(judge, model, messages, a, b, swap_guard=False))
    v2 = _norm_verdict(pairwise_judge(judge, model, messages, b, a, swap_guard=False))
    v2 = {"A": "B", "B": "A", "tie": "tie"}[v2]      # un-swap: "A" there means b won
    return v1 if v1 == v2 else "tie"

judge = fw_client(API_KEY, base_url=f"{API_BASE.rstrip('/')}/inference/v1")
# One raw call before any sampling: pairwise_judge turns every judge error into
# "tie", so an unreachable JUDGE_MODEL would otherwise surface only as an
# all-ties result after minutes of generation.
judge.chat.completions.create(model=JUDGE_MODEL, max_tokens=8,
                              messages=[{"role": "user", "content": "Say OK."}])
print(f"judge {JUDGE_MODEL.split('/')[-1]} is reachable")
subjects = [r for r in eval_rows if prompt_messages(r)][:N_WINRATE]
print(f"win-rate over {len(subjects)} held-out prompts ({2 * len(subjects)} samples)")

# Both adapters must be serving before the sequential sampling below. warm_up() is
# defined in section 10 -- run that cell first.
assert warm_up(reference_client, "base (reference)") and warm_up(tuned_client, "tuned"), (
    "samplers are not serving; re-run this cell (the pool loads adapters on demand)"
)
print()

wins = losses = ties = 0
# Count skips PER SIDE. If one model is skipped more than the other, the
# surviving sample is biased -- e.g. dropping the tuned model's long answers
# would silently measure only the prompts where it happened to be concise.
skips = {"base": {}, "tuned": {}}

for n, row in enumerate(subjects, 1):
    msgs = prompt_messages(row)
    base_ans, base_st = generate(reference_client, msgs)
    tuned_ans, tuned_st = generate(tuned_client, msgs)
    if base_st != "ok":
        skips["base"][base_st] = skips["base"].get(base_st, 0) + 1
    if tuned_st != "ok":
        skips["tuned"][tuned_st] = skips["tuned"].get(tuned_st, 0) + 1
    if base_st != "ok" or tuned_st != "ok":
        print(f"  [{n}/{len(subjects)}] skipped  (base: {base_st} | tuned: {tuned_st})")
        continue

    # A = base, B = tuned  ->  "B" means the tuned model won.
    # Defensive: this is shared code and the judge is a live model. A malformed
    # verdict should cost one prompt, not the whole run.
    try:
        verdict = judge_pair(judge, JUDGE_MODEL, msgs, base_ans, tuned_ans)
    except Exception as e:
        print(f"  [{n}/{len(subjects)}] judge error ({type(e).__name__}) -- counting as tie")
        verdict = "tie"
    wins += verdict == "B"
    losses += verdict == "A"
    ties += verdict == "tie"
    print(f"  [{n}/{len(subjects)}] {'TUNED' if verdict == 'B' else 'BASE' if verdict == 'A' else 'tie'}")

decisive = wins + losses
win_rate = wins / decisive if decisive else 0.0
skipped = len(subjects) - (wins + losses + ties)

print(f"\ntuned won {wins} / lost {losses} / tied {ties} / skipped {skipped}")
for side in ("base", "tuned"):
    if skips[side]:
        print(f"  {side:5s} skips: " + ", ".join(f"{k} x{v}" for k, v in skips[side].items()))
n_base, n_tuned = sum(skips['base'].values()), sum(skips['tuned'].values())
if abs(n_base - n_tuned) > max(2, 0.1 * len(subjects)):
    print(f"  !! ASYMMETRIC SKIPS ({n_base} base vs {n_tuned} tuned) -- the surviving")
    print(f"     sample is biased toward prompts the skipped side handled well.")
    print(f"     Raise MAX_GEN_TOKENS and re-run before trusting this number.")
print(f"\nwin-rate vs base over {decisive} decisive verdicts: {win_rate:.1%}"
      if decisive else "\nno decisive verdicts -- check JUDGE_MODEL")

In [ ]:
%%live
from figures import COLORS

fig, ax = plt.subplots(figsize=(6.5, 4))
bars = ax.bar(["base\n(frozen reference)", "after DPO"], [1 - win_rate, win_rate],
              color=[COLORS["base"], COLORS["adapter"]], width=.6)
for b, v in zip(bars, [1 - win_rate, win_rate]):
    ax.text(b.get_x() + b.get_width() / 2, v + .02, f"{v:.0%}", ha="center", fontweight="bold")
ax.axhline(.5, ls="--", c="gray", lw=1)
ax.annotate("50% = no difference", (.02, .51), xycoords=("axes fraction", "data"),
            fontsize=8, color="gray")
ax.set_ylim(0, 1.1); ax.set_ylabel("share of decisive judgments")
ax.set_title(f"Tuned vs base on {len(subjects)} held-out prompts, judged by {JUDGE_MODEL.split('/')[-1]}\n"
             f"{wins}-{losses} over {decisive} decisive verdicts "
             f"({ties} ties, {skipped} skipped, {len(subjects)} prompts)")
ax.grid(axis="y", alpha=.3); plt.tight_layout(); plt.show()

**What the figure says.** The bar is the tuned model's share of the *decisive* verdicts only. With 32 held-out prompts and the swap guard turning every disagreement into a tie, a handful of decisive verdicts is typical, so read this as a check that the measurement works, not as an effect size; the number you would publish comes from 200+ prompts.


## 12. Ship it (optional)

> **Live.** The next code cell starts with `%%live` and calls the paid serverless trainer. Its real output appears only with `RUN_LIVE = True`; by default it prints a skip notice.

A session snapshot dies with the kernel; promotion is how the adapter outlives it.

**The knob.** The next cell has a second gate on top of `RUN_LIVE`: `PROMOTE = False` by default, so even
a live run registers nothing and prints a skip line. Set `PROMOTE = True` and choose `OUTPUT_MODEL_ID`, the
name the adapter gets under your account (`accounts/<account>/models/<OUTPUT_MODEL_ID>`). Registering is
free; money is spent only when you deploy the model, which needs the `firectl` CLI
([install](https://docs.fireworks.ai/tools-sdks/firectl/firectl)) and is not done in this notebook.

Promotion copies a checkpoint **out of** this session into the model catalog as
`accounts/<account>/models/<output_model_id>`.

**The promotion *call* is session-scoped; the model it produces is not.** The sampler checkpoint it
reads from dies with the session, so this cell has to run now, but what you get is durable.

**A promoted model is not running.** It is a registered LoRA adapter, no GPUs and no endpoint. Serving
it is a separate step. Today that means a dedicated deployment of the base model with addons enabled
(`firectl deployment create accounts/fireworks/models/glm-5p3-flash --enable-addons`, then
`firectl model load-lora <model> --deployment <id>`), billed per replica-hour. Serverless adapter
serving for GLM 5.3 Flash, where the adapter is called by name and billed per token, is what the
gallery is built for; check the docs for its current status.

```
session snapshot  ->  usable now, this kernel only   (sections 10-11 sample this)
save_state ref    ->  portable, resumable            (section 6 writes these)
promoted model    ->  durable, nothing running        (this cell)
deployment        ->  live endpoint                    (not in this notebook)
```

In [ ]:
%%live
PROMOTE = False              # True -> register a durable model (does NOT deploy it)
OUTPUT_MODEL_ID = "dpo-glm53-flash-demo"

if PROMOTE:
    from fireworks.training.sdk import FireworksClient

    def _find_promotable(checkpoints, final_name, run_id):
        # This run's final sampler row, matched by the server-owned id prefix.
        prefixes = [final_name] + ([f"{run_id}-{final_name}"] if run_id else [])
        def label(c): return str(c.get("name", "")).rstrip("/").split("/")[-1]
        return next((c for c in checkpoints if c.get("promotable")
                     and any(label(c) == p or label(c).startswith(p + "-") for p in prefixes)), None)

    def control_plane_base_url(base_url: str) -> str:
        root = base_url.rstrip("/")
        suffix = "/training/v1/serverless"
        return root[: -len(suffix)] if root.endswith(suffix) else root

    cp = FireworksClient(api_key=API_KEY, base_url=control_plane_base_url(API_BASE))
    try:
        # The training SESSION id lives on the service (a "ts-..." id), NOT on
        # training_client.session_id -- that one only disambiguates snapshot names.
        session = (service.training_session_name
                   or f"accounts/{cp.account_id}/trainingSessions/{service.training_session_id}")
        checkpoints = cp.list_training_session_checkpoints(session)
        print("promotable checkpoints:", [c.get("name") for c in checkpoints if c.get("promotable")])
        # The list spans the whole session, oldest first. Match THIS run's
        # snapshot by run id, so a re-run in the same session can't promote
        # an earlier run's dpo-final.
        match = _find_promotable(checkpoints, "dpo-final", getattr(training_client, "run_id", None))
        if match:
            model = cp.promote_session_checkpoint(
                name=match["name"], output_model_id=OUTPUT_MODEL_ID, base_model=BASE_MODEL)
            print("promoted:", model)
        else:
            print("dpo-final is not listed as promotable yet")
    finally:
        cp.close()
else:
    print("skipped. set PROMOTE = True to register the checkpoint as a durable model.")

## 13. Cleanup

> **Live.** The next code cell starts with `%%live` and calls the paid serverless trainer. Its real output appears only with `RUN_LIVE = True`; by default it prints a skip notice.

Serverless has no deployment to delete. This is the entire teardown story.

In [ ]:
%%live
for c in (reference_client, tuned_client):
    try:
        c.close()
    except Exception:
        pass
print("closed sampling clients. nothing left running.")

## What to take away

**Did training help?** DPO alone, no: on the held-out briefs the v1 adapter scored 0.30 on the detector
against base's 0.33, inside the same interval, with more emoji and the same tropes. GRPO on the detector
did move it, 0.33 to 0.79, and cut tropes from 3.4 to 1.0 per 100 words with faithfulness intact.

**Where it failed.** The policy found that the detector rewards plain declarative sentences and converged
on one opener: 72% of its posts start "New blog post:" and the 40 posts share four openers. Every post
passes the judge on its own; the failure is only visible across the set. A diversity term brought back
22 distinct openers of 40 at a detector mean of 0.67; that interval overlaps RL step 20's, so at 40
posts the price of the diversity term is suggested, not established. The reward you pick is the behavior you get.

**Adapter, prompt, or more data?** A one-line prompt that bans the tropes removes them completely
(0.2 per 100 words) and keeps all of base's variety, at a detector score no better than base. If
"anti-slop" means fewer tells, the prompt wins and training bought nothing a prompt could not. If it
means a detector score, RL wins and pays in sameness. More DPO pairs would not change this: the pairs
were ranked by the same detector, so more of them teach the same taste.

**Why the Humanizer chain is the writer the gallery kept.** Run as an editor on base drafts of the same
20 briefs, the [Humanizer adapter](../sft_humanizer/) cuts tropes from 3.8 to 1.3 per 100 words and em
dashes from 78% to 10%, keeps faithfulness and keeps all of base's opener variety; the detector barely
moves (0.30 to 0.38). It removes the tells with no reward to hack, which is the property the RL arm
lost. The `antislop-chain` results on 2026-10-09 hold those numbers.

**One experiment to try next.** Add the opener-diversity term to the reward from step 0 instead of
after the collapse, and put the trope counter into the reward alongside the detector, then re-run the
same 20-brief evaluation with the prompted ban as the control arm. The question it answers is whether
a reward that names the tells can beat the prompt without the sameness.

**And in every run of this loop:** watch `chosen` and `rejected` separately, quote the held-out delta
rather than the loss, and read the text.